# 03 - Retrieval ablation: filter, per-filing search, reranker

Same 81 golden questions, same bge-m3 embeddings, six retrieval variants. Only retrieval changes
(no LLM here), so this notebook is quick: a few minutes on a T4.

- **filter**: guess company and fiscal year from the question and search only those filings;
- **per-filing**: every filing in scope gets its own candidate list (a two-year question sees both years);
- **rerank**: a cross-encoder (bge-reranker-v2-m3) reorders a pool of 30 candidates.

Before running: Runtime > T4 GPU, Colab Secret `HF_TOKEN` (notebook access on).

In [ ]:
import os
import subprocess
import sys

REPO = "agentic-rag-eval"
BRANCH = "p1-agentic-rag"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://github.com/ilaria-cesa/{REPO}.git"], check=True
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import gc
import json

import numpy as np
import pandas as pd
import torch
from google.colab import userdata
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer

from src.eval.golden import load_golden, validate
from src.eval.metrics import retrieval_metrics, summarise
from src.retrieval.embed import embed_texts, load_chunks
from src.retrieval.pipeline import Corpus, RetrievalConfig, retrieve
from src.retrieval.rerank import Reranker

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HF_REPO = "IlariaCesa/sec-10k-chunks"
chunks = load_chunks(hf_hub_download(HF_REPO, "chunks_1600.jsonl", repo_type="dataset"))
embeddings = np.load(
    hf_hub_download(HF_REPO, "embeddings_bge-m3_1600.npy", repo_type="dataset")
).astype(np.float32)
assert len(chunks) == len(embeddings)
questions = [q for q in load_golden("golden/golden_set.jsonl") if q["expected"] == "answer"]
errors, _ = validate(questions, chunks)
assert not errors, errors

embedder = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"dtype": torch.float16})
embedder.max_seq_length = 640
query_vectors = embed_texts(embedder, [q["question"] for q in questions])
del embedder
gc.collect()
torch.cuda.empty_cache()
corpus = Corpus(chunks, embeddings)
reranker = Reranker()
print(len(questions), "answerable questions,", len(chunks), "chunks")

In [ ]:
CONFIGS = [
    RetrievalConfig("A dense (baseline)"),
    RetrievalConfig("B + filter", use_filter=True),
    RetrievalConfig("C + filter + per-filing", use_filter=True, per_filing=True),
    RetrievalConfig("D dense + rerank", rerank=True),
    RetrievalConfig("E filter + rerank", use_filter=True, rerank=True),
    RetrievalConfig(
        "F filter + per-filing + rerank", use_filter=True, per_filing=True, rerank=True
    ),
]
ranked = {}
summaries = {}
for config in CONFIGS:
    rows = []
    for q, vec in zip(questions, query_vectors):
        idx, scope = retrieve(corpus, q["question"], vec, config, reranker)
        ids = [chunks[i]["chunk_id"] for i in idx]
        ranked.setdefault(config.name, {})[q["id"]] = ids
        rows.append(
            {"id": q["id"], "type": q["type"]}
            | retrieval_metrics(q["evidence_chunk_ids"], ids)
            | {"in_scope": scope is not None}
        )
    summaries[config.name] = summarise(rows)

cols = ["hit@1", "hit@5", "hit@10", "recall@5", "all@5", "mrr"]
print(pd.DataFrame({n: s["all"] for n, s in summaries.items()}).T[cols].to_string())

In [ ]:
for metric in ["hit@5", "all@5", "mrr"]:
    print("\n" + metric + " by question type")
    table = pd.DataFrame(
        {n: {t: v[metric] for t, v in s.items() if t != "all"} for n, s in summaries.items()}
    ).T
    print(table.to_string())

In [ ]:
os.makedirs("results", exist_ok=True)
with open("results/retrieval_ablation.json", "w") as f:
    json.dump({"summary": summaries, "ranked": ranked}, f, indent=1)

import shutil

from google.colab import files

shutil.make_archive("retrieval_ablation", "zip", "results")
files.download("retrieval_ablation.zip")

Copy back to the chat: the first table (all questions) and the three tables by question type.
The zip has the ranked chunk ids of every variant, to look at the questions that still fail.